# S4.8 - lag, lead, first, last
**Date completed:** November 2026  
**Status:** Completed  
**Key concept:** Window functions that read neighbouring rows: previous, next, first, last

In [0]:
# ============================================================
# S4.8 - Cell 2
# Navigation / Value Window Functions:
# lag, lead, first, last, nth_value
#
# Goal:
# 1. Understand lag() and lead().
# 2. Understand offset and default values.
# 3. Calculate distance from the previous row.
# 4. Understand first / last with the default frame.
# 5. Understand why last() often needs a full frame.
# 6. Understand RESPECT NULLS vs IGNORE NULLS.
# 7. Use nth_value() as a small extension.
# 8. Reconcile consecutive-order gaps on samples.tpch.
#
# Scope:
# - lag
# - lead
# - first / last
# - NULL behaviour
#
# Small extension:
# - nth_value
#
# Deferred:
# - running sum / rolling average      -> S4.9
# - rolling 7-day RetailPulse exercise -> S4.10
#
# Learning-output rule:
# Show:
#
# current row
# previous / next row
# frame when relevant
# calculated result
# why that result happened
# ============================================================

import pyspark.sql.functions as F
from pyspark.sql import Window


print("=== S4.8: LAG, LEAD, FIRST, LAST ===")
print()

# ============================================================
# Step 1 - Controlled Source Data
#
# Two customers.
#
# Customer 1:
#
# day 1 → amount 10
# day 4 → amount 30
# day 9 → amount 20
#
# Customer 2:
#
# day 2 → amount 100
# day 3 → amount 50
#
# Window:
#
# partitionBy("cust")
# orderBy("day")
#
# Therefore navigation happens independently inside
# each customer.
# ============================================================

df = spark.createDataFrame(
    [
        (1, 1, 10),
        (1, 4, 30),
        (1, 9, 20),

        (2, 2, 100),
        (2, 3, 50)
    ],
    """
    cust INT,
    day INT,
    amount INT
    """
)

w = (Window.partitionBy("cust").orderBy("day"))

print("=== STEP 1: SOURCE ===")

df.orderBy("cust","day").show()

# ============================================================
# Step 2 - lag() and lead()
#
# lag()
#
# → looks BACKWARD in the ordered partition
#
# lead()
#
# → looks FORWARD in the ordered partition
#
# Default offset:
#
# 1 row
#
# Customer 1:
#
# day 1:
# previous = none
# next     = amount 30
#
# day 4:
# previous = amount 10
# next     = amount 20
#
# day 9:
# previous = amount 30
# next     = none
#
# We also calculate:
#
# current day - previous day
#
# to answer:
#
# "How many days since the customer's previous order?"
# ============================================================

navigation = (
    df

    .withColumn("prev_day",F.lag("day").over(w))

    .withColumn("prev_amt",F.lag("amount").over(w))

    .withColumn("next_day",F.lead("day").over(w))

    .withColumn("next_amt",F.lead("amount").over(w))

    .withColumn("prev_or_0",F.lag("amount",1,0).over(w))

    .withColumn("days_since_prev", F.col("day") - F.col("prev_day")))


print()
print("=== STEP 2: LAG AND LEAD ===")

print(
    """
HOW TO READ THIS

current
→ row being evaluated

previous
→ lag(...)

next
→ lead(...)

days_since_prev
→ current day - previous day


At the START of a customer partition:

lag()
→ no previous row
→ NULL


At the END of a customer partition:

lead()
→ no next row
→ NULL
"""
)


for r in (
    navigation
    .orderBy(
        "cust",
        "day"
    )
    .collect()
):

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"previous=(day={r['prev_day']}, amount={r['prev_amt']}) | "
        f"next=(day={r['next_day']}, amount={r['next_amt']}) | "
        f"lag_default_0={r['prev_or_0']} | "
        f"days_since_prev={r['days_since_prev']}"
    )


print()
print(
    "=== KEY OBSERVATION ==="
)


print(
    """
Customer 1:

day 1
→ no previous row
→ lag = NULL
→ lag with default 0 = 0


day 4
→ previous day = 1
→ previous amount = 10
→ 4 - 1 = 3 days


day 9
→ previous day = 4
→ previous amount = 30
→ 9 - 4 = 5 days
"""
)



In [0]:
# ============================================================
# Step 2 - lag() and lead()
#
# lag()
#
# → looks BACKWARD in the ordered partition
#
# lead()
#
# → looks FORWARD in the ordered partition
#
# Default offset:
#
# 1 row
#
# Customer 1:
#
# day 1:
# previous = none
# next     = amount 30
#
# day 4:
# previous = amount 10
# next     = amount 20
#
# day 9:
# previous = amount 30
# next     = none
#
# We also calculate:
#
# current day - previous day
#
# to answer:
#
# "How many days since the customer's previous order?"
# ============================================================

navigation = (
    df
    .withColumn("prev_day",F.lag("day").over(w))

    .withColumn("prev_amt",F.lag("amount").over(w))

    .withColumn("next_day",F.lead("day").over(w))

    .withColumn("next_amt",F.lead("amount").over(w))

    .withColumn("prev_or_0",F.lag("amount",1,0).over(w))

    .withColumn("days_since_prev", F.col("day") - F.col("prev_day")))

print()
print("=== STEP 2: LAG AND LEAD ===")


print(
    """
HOW TO READ THIS

current
→ row being evaluated

previous
→ lag(...)

next
→ lead(...)

days_since_prev
→ current day - previous day


At the START of a customer partition:

lag()
→ no previous row
→ NULL


At the END of a customer partition:

lead()
→ no next row
→ NULL
"""
)


for r in (navigation.orderBy("cust","day").collect()):

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"previous=(day={r['prev_day']}, amount={r['prev_amt']}) | "
        f"next=(day={r['next_day']}, amount={r['next_amt']}) | "
        f"lag_default_0={r['prev_or_0']} | "
        f"days_since_prev={r['days_since_prev']}")

print()
print("=== KEY OBSERVATION ===")

print(
    """
Customer 1:

day 1
→ no previous row
→ lag = NULL
→ lag with default 0 = 0

day 4
→ previous day = 1
→ previous amount = 10
→ 4 - 1 = 3 days


day 9
→ previous day = 4
→ previous amount = 30
→ 9 - 4 = 5 days
"""
)


In [0]:
# ============================================================
# Step 3 - first() and last() With Default Ordered Frame
#
# Window:
#
# partitionBy("cust")
# orderBy("day")
#
# From S4.6 we know the default ordered aggregate frame is:
#
# RANGE BETWEEN
# UNBOUNDED PRECEDING
# AND CURRENT ROW
#
# Therefore:
#
# first(amount)
# → first amount seen from the beginning
#
# last(amount)
# → last value inside the CURRENT frame
#
# In THIS controlled data, day is unique within each customer.
#
# Therefore last_default will equal the current row's amount.
#
# Important:
#
# Do NOT generalize this to tied ORDER BY values.
# ============================================================

default_values = (
    df

    .withColumn("frame_rows",F.collect_list(F.struct("day","amount")).over(w))

    .withColumn("first_amt",F.first("amount").over(w))

    .withColumn("last_default",F.last("amount").over(w)))

print()
print("=== STEP 3: FIRST / LAST WITH DEFAULT FRAME ===")

print(
    """
HOW TO READ THIS

frame
→ rows visible to the current row

first_amt
→ first amount inside that frame

last_default
→ last amount inside that frame

Because the default frame ends at CURRENT ROW:

last() does NOT automatically mean:

"last row of the customer"

It means:

"last value in the current frame"
""")

for r in (default_values.orderBy("cust","day").collect()):

    frame = [
        (
            x["day"],
            x["amount"]
        )
        for x in r["frame_rows"]
    ]

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"frame={frame} | "
        f"first={r['first_amt']} | "
        f"last_default={r['last_default']}"
    )


In [0]:
# ============================================================
# Step 4 - FULL Frame
#
# Requirement:
#
# "Give me the actual first and last amount for the complete
# customer partition on EVERY row."
#
#
# We therefore need:
#
# ROWS BETWEEN
# UNBOUNDED PRECEDING
# AND
# UNBOUNDED FOLLOWING
#
#
# Customer 1:
#
# first = 10
# last  = 20
#
# Customer 2:
#
# first = 100
# last  = 50
#
# nth_value(amount, 2)
#
# means:
#
# "Give me the SECOND row's amount in the frame."
#
# nth_value is 1-based:
#
# 1 → first
# 2 → second
# 3 → third
# ============================================================

w_full = (
    Window
    .partitionBy("cust")
    .orderBy("day")
    .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))

full_values = (
    df

    .withColumn("full_frame",F.collect_list(F.struct("day","amount")).over(w_full))

    .withColumn("first_full",F.first("amount").over(w_full))

    .withColumn("last_full",F.last("amount").over(w_full))

    .withColumn("second_amt",F.nth_value("amount",2).over(w_full)))

print()
print("=== STEP 4: FULL FRAME ===")

for r in (full_values.orderBy("cust","day").collect()):

    frame = [
        (
            x["day"],
            x["amount"]
        )
        for x in r["full_frame"]
    ]

    print(
        f"RESULT "
        f"cust={r['cust']} | "
        f"current=(day={r['day']}, amount={r['amount']}) | "
        f"full_frame={frame} | "
        f"first={r['first_full']} | "
        f"last={r['last_full']} | "
        f"second={r['second_amt']}"
    )


print()
print("=== KEY OBSERVATION: LAST ===")

print(
    """
DEFAULT ordered frame:

start
→ current row

Therefore:

last()
→ last value up to the current row

FULL frame:

start
→ end of customer partition

Therefore:

last()
→ actual final value for the customer
"""
)

In [0]:
# ============================================================
# Step 5 - NULL Behaviour
#
# This is important.
#
# There are TWO different situations:
#
#
# Situation A
#
# The previous row DOES NOT EXIST.
#
# lag(amount, 1, 0)
#
# → returns default 0
#
#
# Situation B
#
# The previous row EXISTS,
# but its amount is NULL.
#
# lag(amount, 1, 0)
#
# → returns NULL
#
#
# The default does NOT replace a real NULL value.
#
#
# We also compare:
#
# first / last
#
# with:
#
# RESPECT NULLS
# vs
# IGNORE NULLS
# ============================================================

null_df = spark.createDataFrame(
    [
        (1, 1, None),
        (1, 2, 10),
        (1, 3, 20),
        (1, 4, None)
    ],
    """
    cust INT,
    day INT,
    amount INT
    """
)


w_null = (
    Window
    .partitionBy("cust")
    .orderBy("day")
)


w_null_full = (
    Window
    .partitionBy("cust")
    .orderBy("day")
    .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))

null_navigation = (null_df

    .withColumn("lag_default_0",F.lag("amount",1,0).over(w_null))

    .withColumn("lead_default_minus1",F.lead("amount",1,-1).over(w_null)))

print()
print("=== STEP 5A: LAG / LEAD DEFAULT vs REAL NULL ===")

for r in (null_navigation.orderBy("day").collect()):

    print(
        f"RESULT "
        f"day={r['day']} | "
        f"amount={r['amount']} | "
        f"lag_default_0={r['lag_default_0']} | "
        f"lead_default_-1={r['lead_default_minus1']}")

print()
print(
    """
IMPORTANT

day 1:

no previous row exists
→ lag(..., default=0)
→ 0


day 2:

previous row exists
but previous amount is NULL

→ lag(..., default=0)
→ NULL

The default is used only when the requested offset row
does not exist. It is not a NULL replacement function.
"""
)


# ------------------------------------------------------------
# first / last NULL handling
# ------------------------------------------------------------

null_first_last = (
    null_df

    .withColumn(
        "first_respect_nulls",
        F.first(
            "amount",
            ignorenulls=False
        ).over(
            w_null_full
        )
    )

    .withColumn(
        "first_ignore_nulls",
        F.first(
            "amount",
            ignorenulls=True
        ).over(
            w_null_full
        )
    )

    .withColumn(
        "last_respect_nulls",
        F.last(
            "amount",
            ignorenulls=False
        ).over(
            w_null_full
        )
    )

    .withColumn(
        "last_ignore_nulls",
        F.last(
            "amount",
            ignorenulls=True
        ).over(
            w_null_full
        )
    )
)


print()
print(
    "=== STEP 5B: RESPECT NULLS vs IGNORE NULLS ==="
)


(
    null_first_last

    .select(
        "cust",
        "day",
        "amount",
        "first_respect_nulls",
        "first_ignore_nulls",
        "last_respect_nulls",
        "last_ignore_nulls"
    )

    .orderBy("day")

    .show()
)


print(
    """
Full customer frame:

amounts:

NULL, 10, 20, NULL


RESPECT NULLS:

first → NULL
last  → NULL


IGNORE NULLS:

first → 10
last  → 20


Use IGNORE NULLS only when skipping NULL values matches
the business requirement.
"""
)

In [0]:
# ============================================================
# Step 6 - Real Data: Consecutive Order Gaps
#
# Business question:
#
# "How many days passed since this customer's previous order?"
#
#
# Data:
#
# samples.tpch.orders
#
# Same 3 customers used in earlier exercises.
#
#
# Window:
#
# partitionBy customer
#
# orderBy:
# order date
# order key
#
#
# order key is a deterministic tie-breaker for orders
# sharing the same date.
#
#
# First order of each customer:
#
# lag(orderdate)
# → NULL
#
# Therefore:
#
# expected NULL gaps
# =
# number of customers
#
#
# expected non-NULL gaps
# =
# order rows - customer count
# ============================================================

o = (
    spark.table("samples.tpch.orders")

    .filter(F.col("o_custkey").isin(721180,382414,299701))

    .select("o_custkey","o_orderkey","o_orderdate"))


w_real = (
    Window.partitionBy("o_custkey")
    .orderBy("o_orderdate","o_orderkey"))

gaps = (
    o.withColumn("previous_order_date",
        F.lag("o_orderdate"
        ).over(w_real))

    .withColumn("gap_days",F.datediff("o_orderdate","previous_order_date")))


n_orders = (gaps.count())


n_customers = (gaps.select("o_custkey").distinct().count())

n_gap = (gaps.filter(F.col("gap_days").isNotNull()).count())

n_null = (gaps.filter(F.col("gap_days").isNull()).count())

negative_gaps = (gaps.filter(F.col("gap_days") < 0).count())

largest_gaps = (
    gaps

    .groupBy("o_custkey")

    .agg(F.max("gap_days").alias("max_gap_days"))

    .orderBy("o_custkey").collect())

print()
print(
    "=== STEP 6: REAL DATA - ORDER GAPS ===")

print(f"RESULT order rows             : "f"{n_orders}")

print(f"RESULT customers              : "f"{n_customers}")

print(f"RESULT rows with gap          : "f"{n_gap}")

print(f"RESULT rows with NULL gap     : "f"{n_null}")

print(f"RESULT orders - customers == rows with gap: "f"{n_orders - n_customers == n_gap}")

print(f"RESULT NULL gaps == customers: "f"{n_null == n_customers}")

print(f"RESULT negative gaps          : "f"{negative_gaps}")

print()
print("Largest order gap by customer:")

for r in largest_gaps:

    print(
        f"RESULT "
        f"cust={r['o_custkey']} | "
        f"largest_gap_days={r['max_gap_days']}")

In [0]:
# ============================================================
# Step 7 - Controlled Checkpoint
# ============================================================

print()
print("=== STEP 7: S4.8 CHECKPOINT ===")

# ------------------------------------------------------------
# Controlled lag/lead expectations.
# ------------------------------------------------------------

cust1_navigation = [
    (
        r["day"],
        r["amount"],
        r["prev_amt"],
        r["next_amt"],
        r["days_since_prev"]
    )
    for r in (
        navigation
        .filter(
            F.col("cust") == 1
        )
        .orderBy("day")
        .collect()
    )
]

expected_cust1 = [
    (1, 10, None, 30, None),
    (4, 30, 10, 20, 3),
    (9, 20, 30, None, 5)]

print("RESULT customer 1 navigation:",cust1_navigation)

print("RESULT navigation matched:", cust1_navigation == expected_cust1)

# ------------------------------------------------------------
# Full-frame last-value expectations.
# ------------------------------------------------------------

full_last_values = sorted({(r["cust"],r["last_full"])for r in full_values.collect()})

print("RESULT full-frame last values:",full_last_values)

# ------------------------------------------------------------
# Real-data reconciliation.
# ------------------------------------------------------------

print("RESULT real gap reconciliation:", n_orders - n_customers == n_gap)

print("RESULT one NULL gap per customer:", n_null == n_customers)

print("RESULT no negative gaps:", negative_gaps == 0)

# Controlled assertions.

assert (cust1_navigation == expected_cust1)

assert (
    full_last_values
    ==
    [(1, 20),(2, 50)])

assert (n_orders - n_customers == n_gap)

assert (n_null == n_customers)

assert (negative_gaps == 0)

print("RESULT all S4.8 checks passed.")

In [0]:

# ============================================================
# Step 8 - Final Learning Summary
# ============================================================

print()
print(
    "=== FINAL LEARNING SUMMARY ==="
)


print("""
LAG

Looks backward inside the ordered partition.

lag(col)
→ previous row

lag(col, 2)
→ two rows back


LEAD

Looks forward inside the ordered partition.

lead(col)
→ next row


--------------------------------------------

DEFAULT VALUE

lag(col, 1, 0)

The default is used when the requested previous
row DOES NOT EXIST.

It does NOT replace a NULL value from an existing row.


--------------------------------------------

FIRST

first(value)

Returns the first value inside the current window frame.


LAST

last(value)

Returns the last value inside the current window frame.


IMPORTANT

last()
does NOT automatically mean:

"last value in the complete partition"

The result depends on the FRAME.


--------------------------------------------

DEFAULT ORDERED FRAME

With:

partitionBy(...)
orderBy(...)

the frame normally ends at CURRENT ROW.

Therefore last() can represent the latest value
seen so far rather than the final partition value.


--------------------------------------------

FULL FRAME

ROWS BETWEEN
UNBOUNDED PRECEDING
AND
UNBOUNDED FOLLOWING

Now first / last can see the complete partition.


--------------------------------------------

NTH_VALUE

nth_value(col, 2)

→ second value in the window frame

Counting begins at 1.


--------------------------------------------

NULLS

RESPECT NULLS

NULL values participate.


IGNORE NULLS

NULL values are skipped.


Use IGNORE NULLS only when that matches the
business requirement.


--------------------------------------------

COMMON BUSINESS USES

lag
→ previous transaction
→ previous status
→ previous order date
→ period-over-period comparison


lead
→ next event
→ next appointment
→ next status


first
→ first event / first order


last with full frame
→ final event / latest value


nth_value
→ Nth event/value inside a sequence


--------------------------------------------

CORE RULE

lag / lead
→ navigate relative to CURRENT ROW


first / last / nth_value
→ depend on the WINDOW FRAME


partitionBy
→ whose history?


orderBy
→ what sequence?


frame
→ what portion of that sequence is visible?
""")

print("=== S4.8 CELL 2 COMPLETED ===")

# S4.8 — Key Takeaways: lag, lead, first, last

## 1. Neighbour Functions

```python
w = (
    Window
    .partitionBy("cust")
    .orderBy("day")
)

F.lag("amount").over(w)          # previous row
F.lead("amount").over(w)         # next row
F.lag("amount", 1, 0).over(w)    # default 0 if previous row does not exist
```

### lag

```text
current row
← previous row
```

`lag()` looks backward inside the ordered partition.

### lead

```text
current row
→ next row
```

`lead()` looks forward inside the ordered partition.

The first row of a partition has no previous row:

```text
lag()
→ NULL
```

The last row has no next row:

```text
lead()
→ NULL
```

---

## 2. lag / lead Default Values

Example:

```python
F.lag(
    "amount",
    1,
    0
).over(w)
```

The default value is used only when the requested offset row does not exist.

These two cases are different:

```text
Previous row does not exist
→ default value

Previous row exists
but its amount is NULL
→ NULL
```

Therefore:

```text
lag(col, 1, 0)
```

is not a general NULL-replacement operation.

---

## 3. Observed lag / lead Behaviour

Customer 1:

```text
(day=1, amount=10)
(day=4, amount=30)
(day=9, amount=20)
```

Observed:

```text
day 1
current amount = 10
previous       = NULL
next           = 30
lag default    = 0
days since previous = NULL


day 4
current amount = 30
previous       = 10
next           = 20
lag default    = 10
days since previous = 3


day 9
current amount = 20
previous       = 30
next           = NULL
lag default    = 30
days since previous = 5
```

Compact result:

```text
lag:
NULL, 10, 30

lead:
30, 20, NULL

lag with default 0:
0, 10, 30

days since previous:
NULL, 3, 5
```

---

## 4. first and last Depend on the Frame

`first()` and `last()` operate on the rows visible inside the current window frame.

With:

```python
Window \
    .partitionBy("cust") \
    .orderBy("day")
```

the ordered default frame ends at the current ordering position.

Therefore:

```text
first()
→ first value visible in the current frame

last()
→ last value visible in the current frame
```

`last()` does **not** automatically mean:

```text
the final value in the entire customer partition
```

In our controlled dataset, `day` is unique within each customer, so the last value in the default frame happens to be the current row's amount.

Customer 1 observed:

```text
day 1 → last_default = 10
day 4 → last_default = 30
day 9 → last_default = 20
```

With tied `ORDER BY` values, peer rows can affect the current frame, so do not generalize this as “last always equals current row.”

---

## 5. Full Partition Frame

To make every row see the complete partition:

```python
w_full = (
    w
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )
)
```

Now:

```python
F.first("amount").over(w_full)
F.last("amount").over(w_full)
```

can see the complete customer sequence.

### Customer 1

```text
10, 30, 20

first = 10
last  = 20
```

Every customer-1 row therefore receives:

```text
last_full = 20
```

### Customer 2

```text
100, 50

first = 100
last  = 50
```

Every customer-2 row receives:

```text
last_full = 50
```

---

## 6. nth_value

```python
F.nth_value(
    "amount",
    2
).over(w_full)
```

means:

```text
second value in the window frame
```

Counting starts at:

```text
1
```

Observed:

```text
Customer 1:
10, 30, 20

second value
→ 30


Customer 2:
100, 50

second value
→ 50
```

Spark documents that if the frame contains fewer than `n` rows:

```text
nth_value(..., n)
→ NULL
```

We did not separately test that case in our workspace.

---

## 7. NULL Handling

Controlled values:

```text
NULL, 10, 20, NULL
```

### RESPECT NULLS

Default behaviour:

```text
first → NULL
last  → NULL
```

because the first and final values in the complete frame are NULL.

### IGNORE NULLS

Using:

```python
F.first(
    "amount",
    ignorenulls=True
)

F.last(
    "amount",
    ignorenulls=True
)
```

produced:

```text
first → 10
last  → 20
```

Use `IGNORE NULLS` only when skipping NULL values matches the business requirement.

---

## 8. lag Default vs Real NULL

Controlled example:

```text
day 1 → amount NULL
day 2 → amount 10
```

For day 1:

```text
no previous row exists

lag(amount, 1, 0)
→ 0
```

For day 2:

```text
previous row exists
previous amount = NULL

lag(amount, 1, 0)
→ NULL
```

Therefore:

```text
default
≠
NULL replacement
```

---

## 9. Typical Business Uses

### lag

```text
previous transaction
previous order
previous status
previous balance
previous measurement
period-over-period comparison
```

### lead

```text
next transaction
next status
next appointment
next event
```

### first

```text
first order
first interaction
initial status
```

### last with complete frame

```text
final event
final status
latest value in the ordered partition
```

### nth_value

```text
second event
third transaction
Nth value in an ordered sequence
```

---

## 10. Real Data — samples.tpch

Three customers were tested.

Window:

```text
partitionBy:
o_custkey

orderBy:
o_orderdate
o_orderkey
```

`o_orderkey` gives a deterministic sequence when multiple orders share the same date.

For each order:

```text
previous_order_date
=
lag(o_orderdate)
```

Then:

```text
gap_days
=
current order date
-
previous order date
```

Observed:

```text
orders            = 115
customers         = 3

rows with gap     = 112
NULL gaps         = 3
```

Reconciliation:

```text
orders - customers
=
115 - 3
=
112

rows with gap
=
112
```

Therefore:

```text
orders - customers == rows with gap
→ True
```

Each customer's first order had no previous order:

```text
NULL gaps = customers
3 = 3
```

---

## 11. Largest Observed Order Gaps

```text
customer 299701
→ 190 days

customer 382414
→ 251 days

customer 721180
→ 234 days
```

These are the maximum gaps between consecutive ordered orders for the three tested customers.

---

## 12. Negative Gap Check

Observed:

```text
negative gaps = 0
```

However, this is not a meaningful source-data quality result here.

Why?

The window itself is ordered by:

```text
o_orderdate ASC
o_orderkey ASC
```

Therefore the previous ordered date cannot normally be later than the current ordered date.

So:

```text
negative gaps = 0
```

is largely a consequence of the window ordering rather than proof that the original source contains no date-quality issues.

---

## 13. Not Fully Tested in Our Workspace

### nth_value with fewer than n rows

Not executed in our notebook.

Documented Spark behaviour:

```text
frame contains fewer than n rows
→ nth_value returns NULL
```

### Same-day consecutive orders

We did not specifically observe or isolate a same-day consecutive-order case.

If two consecutive ordered rows have the same date:

```text
datediff(
    current_date,
    previous_date
)

→ 0
```

That would be a valid non-NULL gap of zero days.

### Negative gaps

Observed `0`, but—as explained above—this follows from ordering the window by ascending date and is not independently useful as a data-quality test.

---

## 14. Interview Q&A

### Q1. How do you get the previous row's value?

Use:

```python
F.lag("col").over(
    Window
    .partitionBy(...)
    .orderBy(...)
)
```

---

### Q2. How do you get the next row's value?

Use:

```python
F.lead("col").over(w)
```

---

### Q3. What does the lag default do?

It supplies a value when the requested previous row does not exist.

It does not replace NULL stored in an existing row.

---

### Q4. Why can last() return something other than the partition's final value?

Because `last()` operates on the current window frame.

If the frame ends at the current ordering position, future rows are not visible.

---

### Q5. How do you get the true final value for every row?

Use a full frame:

```text
UNBOUNDED PRECEDING
to
UNBOUNDED FOLLOWING
```

with deterministic ordering.

---

### Q6. How do you skip NULLs in first or last?

Use:

```python
ignorenulls=True
```

only when skipping NULL values matches the business requirement.

---

### Q7. What does nth_value(col, 2) mean?

The second value in the window frame.

Counting begins at `1`.

---

### Q8. How would you calculate days since a customer's previous order?

```text
partitionBy(customer)
orderBy(order_date, tie_breaker)

lag(order_date)
      ↓
datediff(
    current_order_date,
    previous_order_date
)
```

---

## 15. Core Understanding

```text
lag
→ look backward

lead
→ look forward
```

They navigate relative to the current row.

```text
first
last
nth_value
→ depend on what the frame can see
```

And:

```text
partitionBy
→ whose history?

orderBy
→ what sequence?

frame
→ what part of the sequence is visible?
```

### Core Rule

```text
Navigation functions
→ understand position

Value functions
→ understand frame

Both
→ require meaningful ordering
```

**S4.8 — lag, lead, first, last COMPLETED.**